# ProjectMem - Full Benchmark on Colab

Chạy full benchmark với tất cả scenario types để đánh giá pipeline.

In [ ]:
# Mount Drive để lưu kết quả (tùy chọn)
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/<your-username>/ProjectMem.git
%cd ProjectMem
!pip install -q -r requirements.txt  # nếu có, nếu không thì cài thư viện cần thiết
!pip install -q pyyaml transformers torch tensorflow  # cài đặt dependencies

In [ ]:
import sys, json, time
sys.path.insert(0, 'src')

from src.pipeline.multi_agent_pipeline import MultiAgentPipeline, load_benchmark
from src.benchmarks.scenario_types import (
    generate_overwrite_scenario,
    generate_keep_multiple_versions_scenario,
    generate_merge_scenario,
    generate_reject_scenario,
    generate_stale_read_scenario,
    generate_structured_merge_scenario,
    generate_multi_agent_scenario,
)

In [ ]:
# Tạo full benchmark: 3 scenarios mỗi loại
benchmark = []
subjects = ['user_001', 'user_002', 'user_003']
predicates = ['lang', 'interests', 'preferences', 'ssn', 'email', 'location', 'job', 'hobby']

for i, subj in enumerate(subjects):
    bench_id = f'overwrite_{i:03d}'
    benchmark.append(generate_overwrite_scenario(bench_id, subj, predicates[i % len(predicates)]))

for i, subj in enumerate(subjects):
    bench_id = f'keep_multi_{i:03d}'
    benchmark.append(generate_keep_multiple_versions_scenario(bench_id, subj, predicates[(i+1) % len(predicates)]))

for i, subj in enumerate(subjects):
    bench_id = f'merge_{i:03d}'
    benchmark.append(generate_merge_scenario(bench_id, subj, predicates[(i+2) % len(predicates)]))

for i, subj in enumerate(subjects):
    bench_id = f'reject_{i:03d}'
    benchmark.append(generate_reject_scenario(bench_id, subj, predicates[(i+3) % len(predicates)]))

for i, subj in enumerate(subjects):
    bench_id = f'stale_{i:03d}'
    benchmark.append(generate_stale_read_scenario(bench_id, subj, predicates[(i+4) % len(predicates)]))

for i, subj in enumerate(subjects):
    bench_id = f'struct_merge_{i:03d}'
    benchmark.append(generate_structured_merge_scenario(bench_id, subj, predicates[(i+5) % len(predicates)]))

print(f'Total scenarios: {len(benchmark)}')


In [ ]:
# Chạy full benchmark với conflict_aware mode
pipeline = MultiAgentPipeline(mode='conflict_aware', persistence_path='tmp_colab_store.jsonl')

results = []
start = time.time()

for idx, sc in enumerate(benchmark):
    logs = pipeline.run_scenario(sc, enable_retrieval_eval=False)
    results.append({
        'scenario_id': sc['scenario_id'],
        'scenario_type': sc.get('scenario_type', 'unknown'),
        'state_match': logs['metrics']['state_match'],
        'num_conflicts': logs['metrics']['num_conflicts'],
        'decisions': [(d['step'], d['resolution_action']) for d in logs['arbitration_decisions']],
    })
    if (idx+1) % 5 == 0:
        print(f'Processed {idx+1}/{len(benchmark)} scenarios...')

elapsed = time.time() - start
print(f'Done in {elapsed:.1f}s')

In [ ]:
# Tổng hợp kết quả
total = len(results)
passed = sum(1 for r in results if r['state_match'])
print(f'\n=== Benchmark Results ===')
print(f'Total: {total}, Passed: {passed}, Failed: {total-passed}')
print(f'Accuracy: {passed/total*100:.1f}%\n')

# Chi tiết theo loại
from collections import defaultdict
by_type = defaultdict(lambda: {'total':0, 'passed':0})
for r in results:
    t = r['scenario_type']
    by_type[t]['total'] += 1
    if r['state_match']:
        by_type[t]['passed'] += 1

for t, stat in sorted(by_type.items()):
    acc = stat['passed']/stat['total']*100 if stat['total'] else 0
    print(f'{t:30s}: {stat["passed"]}/{stat["total"]} ({acc:.1f}%)')

# Lưu kết quả
with open('benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to benchmark_results.json')

In [ ]:
# Hiện các scenarios bị fail (nếu có)
failed = [r for r in results if not r['state_match']]
if failed:
    print(f'\n=== Failed Scenarios ({len(failed)}) ===')
    for r in failed:
        print(f"  {r['scenario_id']} ({r['scenario_type']}): decisions={r['decisions']}")
else:
    print('\nAll scenarios passed!')